# Challenge 3: Groundedness and tool-selection evals

Two checks decide whether an internal assistant can be trusted before release:

1. **Groundedness:** does the answer stick to the sources the agent retrieved? An LLM judge reads the answer and the retrieved text.
2. **Tool selection:** did the agent use the right tools, no more and no fewer? It should search before answering policy questions, and update a ticket only when asked.

The run you make here is the **baseline** that challenge 4 improves on.

**One cell was blank in the workshop. Here it is filled in, so every cell runs.**

In [ ]:
import os
import sys

for path in ("..", "../lab"):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv("../.env", override=True)

from langsmith import Client, uuid7

import lab_eval
from eval_lab.agent import SYSTEM_PROMPT_V1, TOOLS, build_agent, extract_sources, extract_tool_calls

client = Client()
DATASET = lab_eval.dataset_name()
print("project:", os.environ["LANGSMITH_PROJECT"])
print("dataset:", DATASET)

## 1. Groundedness (LLM-as-judge)

The judge gets the question, the agent's answer, and the tool results it retrieved: documents and ticket data (`NONE` if it called no tools). It marks `score=True` only if every claim in the answer is supported by that text. An answer that says "I couldn't find it" is grounded when the sources really don't contain it.

`judge_model` is the small, cheap model set by `JUDGE_MODEL`. `with_structured_output` forces its reply into the schema.

In [ ]:
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage

from utils.models import judge_model


class GroundednessGrade(TypedDict):
    '''Score whether every claim in the answer is supported by the retrieved sources.'''
    score: bool       # True if grounded, False otherwise
    reasoning: str    # one-sentence explanation


GROUNDEDNESS_PROMPT = (
    "You are checking whether an internal assistant's answer is grounded in its sources.\n"
    "You will see a question, the assistant's answer, and the tool results it retrieved (knowledge-base documents and ticket data).\n"
    "Mark score=True only if every factual claim in the answer is supported by the sources.\n"
    "Mark score=False if the answer states facts, numbers or policies that the sources do not contain, "
    "or if the sources are NONE and the answer still asserts specific facts.\n"
    "An answer that says the information could not be found is grounded when the sources do not contain it.\n"
    "Give one short sentence of reasoning."
)

grounding_judge = judge_model.with_structured_output(GroundednessGrade)


def groundedness(inputs, outputs):
    sources = "\n\n".join(outputs.get("sources") or []) or "NONE"
    grade = grounding_judge.invoke([
        SystemMessage(content=GROUNDEDNESS_PROMPT),
        HumanMessage(content=(
            f"Question: {inputs['query']}\n\n"
            f"Assistant answer: {outputs['response']}\n\n"
            f"Sources:\n{sources}"
        )),
    ])
    return {"key": "groundedness", "score": int(grade["score"]), "comment": grade["reasoning"]}

## 2. Tool selection

Each example's `outputs.trajectory` lists the tools the agent should use. The evaluator compares it to the tools the agent actually called, as sets, so a skipped search and an unrequested ticket update both fail.

The next cell holds this code:

```python
def tool_selection(outputs, reference_outputs):
    used = set(outputs["trajectory"])
    expected = set(reference_outputs["trajectory"])
    return {"key": "tool_selection", "score": int(used == expected)}
```

In [ ]:
# Solution. In the workshop this cell was blank and you wrote it.
def tool_selection(outputs, reference_outputs):
    used = set(outputs["trajectory"])
    expected = set(reference_outputs["trajectory"])
    return {"key": "tool_selection", "score": int(used == expected)}

## 3. Run the baseline

Same dataset version as challenge 2, same target, three evaluators. The cheap `keyword_match` rides along so you can compare it with the two new ones. This takes about two minutes.

In [ ]:
from eval_lab.evaluators import keyword_match, make_run_agent

results = client.evaluate(
    make_run_agent(build_agent()),
    data=lab_eval.pinned_examples(client),
    evaluators=[groundedness, tool_selection, keyword_match],
    experiment_prefix="v1-baseline",
    max_concurrency=4,
)
lab_eval.compare(prefixes=("first-eval", "v1-baseline"))

## 4. Read the results

Open the `v1-baseline` experiment in LangSmith. Sort by `groundedness`, open a row that scored 0, and read the judge's **comment**. Then open the row's trace and look at the child spans: was a search ever called?

**Result:** `keyword_match` looks acceptable, but `groundedness` and `tool_selection` are lower. The agent gives answers that sound right without using the knowledge base. For a business that has to trust its answers that is the failure that matters, and a string check never saw it.